# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Soham334/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

# 1. My lane as an ML task

## Task type: Ranking

My chosen lane is **Refresh / Content Opportunity Scoring**.

The goal is to rank content pages by how strongly they should be considered for review or refresh.

This is best framed as a **ranking problem** because the practical question is not simply whether a page is "good" or "bad." The content team needs to know **which pages should be reviewed first** when time and resources are limited.

The model would produce a priority score for each content item. Pages with higher scores would appear higher in a review queue.

### Research question

> Which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?

The unit being ranked is an individual content item.

The ranking output supports a practical content operation: deciding which pages deserve attention first.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

# 2. Target or proxy

## Target / proxy: likelihood that a page is declining

For this first framing exercise, I will use **decline risk as a proxy for refresh priority**.

The starter pipeline defines the decline label as:

`is_declining_label = (trend_direction == "down")`

This gives the model a concrete outcome to learn from.

A model could estimate the probability that a content item belongs to the declining group. That probability can then be converted into a ranking score:

- higher predicted decline probability → higher review priority
- lower predicted decline probability → lower immediate review priority

This is only a proxy for the broader business decision. A declining page is not automatically a page that should be refreshed. Other considerations may matter when an actual content team makes the final decision.

### Leakage consideration

I would not use `trend_direction` or `trend_pct` as model features because they directly define or reveal the decline label.

I would also avoid product-generated flags such as `health_score`, `is_quick_win`, and other action/health flags when building the model, because FlyRank identifies these as leakage risks.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

# 3. Success metric

## Primary metric: Precision@50

I would evaluate the ranking using **Precision@50**.

Precision@50 asks:

> Of the 50 pages ranked highest by the model, how many are actually in the target group?

This metric matches the real workflow because a content team may only have enough time to review a small number of pages first.

For example, if Precision@50 is 0.70, approximately 70% of the top 50 ranked pages would be expected to belong to the declining target group.

The important part is that the model should put useful candidates near the top of the queue, rather than merely achieving good overall classification accuracy.

The starter dataset provides a useful reference point: FlyRank reports a rule baseline Precision@50 of approximately 0.26 and a Random Forest result of approximately 0.74 on the 30k-row starter slice. These numbers are reference results rather than results from my own experiment.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [2]:
import os

print(os.getcwd())

/content


In [4]:
import pandas as pd

# Load the starter dataset directly from this repository.
DATA_URL = (
    "https://raw.githubusercontent.com/"
    "Soham334/flyrank-ml-internship/"
    "main/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)

print("Dataset shape:", df.shape)

# Select a small, readable slice for this lane.
lane_columns = [
    "content_id",
    "client_id",
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "trend_direction",
    "trend_pct",
]

lane_df = df[lane_columns].copy()

print("\nUnit of analysis: one row = one content item/page")
display(lane_df.head(10))

Dataset shape: (30000, 44)

Unit of analysis: one row = one content item/page


,content_id,client_id,search_volume,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,content_age_days,days_since_last_update,ctr,avg_position,engagement_rate,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,3803,29,22,17,187,20,0.76,10.6,5.88,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,15320,7,10,9,445,25,0.05,20.3,0.00,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,12581,11,14,11,141,20,0.09,36.5,0.00,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,11751,58,87,78,463,22,0.49,6.2,1.28,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,19140,24,177,145,263,14,0.13,44.0,0.00,down,-34.7
5,content_d4084a4bc775,client_f369cb89fc,720.0,3970,1,4,5,147,20,0.03,8.5,0.00,down,-38.9
6,content_9a34b442b552,client_8722616204,0.0,20,0,1,1,90,20,0.00,7.0,0.00,down,-92.3
7,content_a63219c6e95a,client_19581e27de,590.0,1724,1,28,28,445,22,0.06,21.2,3.57,stable,0.6
8,content_5e6c160719bc,client_6208ef0f77,0.0,32574,29,128,68,90,20,0.09,46.0,5.88,down,-58.8
9,content_c27558df2b0c,client_19581e27de,0.0,1240,2,4,3,257,104,0.16,4.9,0.00,down,-29.2


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### Why ML can beat a fixed rule

A fixed rule could be something like:

> "Review every page whose traffic is declining."

This is easy to understand, but it treats the decision as a single threshold and does not naturally prioritize pages with different combinations of signals.

Content items can differ in search impressions, clicks, sessions, engagement, search position, content age, and freshness.

An ML model can learn relationships across several signals and produce a continuous predicted risk score for each content item.

That score can then be used to rank the review queue.

This is useful when the team has limited capacity. Instead of treating every page that matches a rule equally, the highest-priority candidates can be reviewed first.

ML would therefore support the human decision rather than replace it. The content team would still decide what action to take after reviewing the ranked candidates.

In [5]:
# Create a simple check that the ranking task has the information
# needed to produce a target and a review queue.

df["is_declining_label"] = (
    df["trend_direction"].eq("down")
).astype(int)

print("Rows:", len(df))
print("Declining target rate:", round(df["is_declining_label"].mean(), 3))

# A simple descriptive ranking proxy for this framing exercise:
# declining pages first, then larger search visibility.
review_queue = (
    df.assign(
        is_declining_label=df["trend_direction"].eq("down").astype(int)
    )
    .sort_values(
        ["is_declining_label", "impressions_90d"],
        ascending=[False, False]
    )
)

display(
    review_queue[
        [
            "content_id",
            "impressions_90d",
            "clicks_90d",
            "content_age_days",
            "days_since_last_update",
            "trend_direction",
            "is_declining_label",
        ]
    ].head(10)
)

Rows: 30000
Declining target rate: 0.542


,content_id,impressions_90d,clicks_90d,content_age_days,days_since_last_update,trend_direction,is_declining_label
6653,content_5fe46e04994d,517715,741,537,104,down,1
26844,content_8c19996aa890,509252,785,445,20,down,1
21819,content_4c36c775b818,463103,1889,445,20,down,1
29879,content_1a9e894be2e2,416180,944,482,22,down,1
13537,content_2c2606c5d176,347399,1854,362,104,down,1
26531,content_cb112fce36be,309910,492,126,104,down,1
21565,content_9532f197bbc8,309192,2689,445,104,down,1
27478,content_008fb02c46cb,236803,605,111,20,down,1
23767,content_813e88069237,233561,129,153,104,down,1
26304,content_ff94c9b6b411,228566,89,154,20,down,1


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### My final framing

- **Task type:** Ranking
- **Target / proxy:** Decline risk, represented by `is_declining_label`
- **Success metric:** Precision@50
- **Unit of analysis:** One row = one pseudonymized content item/page
- **Action:** Prioritize content items for human review and possible refresh
- **Why ML:** Multiple content-performance signals can be combined into a ranked priority score instead of relying on one fixed threshold
- **Leakage awareness:** `trend_direction` and `trend_pct` define the target and should not be used as model features
- **Decision role:** The model provides decision support; humans make the final content action decision